# 02 — Preprocessing & Feature Engineering

Ce notebook orchestre l'étape 1 du pipeline :
1. Chargement et nettoyage des données LIAR
2. Regroupement des labels (6 → 3 classes)
3. Calcul des features de métadonnées
4. Construction des représentations TF-IDF + métadonnées
5. Vérification et sauvegarde des données préprocessées

Les fonctions utilisées ici sont définies dans `src/preprocessing.py` et `src/features.py`.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import plotly.express as px
import os

from src.preprocessing import (
    load_liar, preprocess_liar, print_summary, LABEL_ENCODE
)
from src.features import TfidfFeatures, CombinedFeatures, get_X_y

os.makedirs("data/processed", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("outputs/figures", exist_ok=True)

print("Imports OK")

Imports OK


## 1. Chargement et preprocessing LIAR

In [2]:
# Chargement des 3 splits bruts
train_raw, valid_raw, test_raw = load_liar(data_dir="data/raw")

# Pipeline complet de preprocessing sur chaque split
train = preprocess_liar(train_raw)
valid = preprocess_liar(valid_raw)
test  = preprocess_liar(test_raw)

# Résumés
print_summary(train, "Train")
print_summary(valid, "Valid")
print_summary(test,  "Test")

LIAR chargé — train: 10240, valid: 1284, test: 1267

=== Résumé — Train (10240 lignes) ===
   nuanced    :  3768  (36.8%)
   real       :  3638  (35.5%)
   fake       :  2834  (27.7%)
   Texte vide : 0 lignes

=== Résumé — Valid (1284 lignes) ===
   nuanced    :   485  (37.8%)
   real       :   420  (32.7%)
   fake       :   379  (29.5%)
   Texte vide : 0 lignes

=== Résumé — Test (1267 lignes) ===
   nuanced    :   477  (37.6%)
   real       :   449  (35.4%)
   fake       :   341  (26.9%)
   Texte vide : 0 lignes


## 2. Vérification du nettoyage textuel

In [3]:
# Comparaison texte brut vs texte nettoyé sur quelques exemples
print("=== Exemples de nettoyage ===")
for i in [0, 1, 2]:
    print(f"\n--- Exemple {i+1} ---")
    print(f"Brut    : {train['statement'].iloc[i]}")
    print(f"Nettoyé : {train['statement_clean'].iloc[i]}")
    print(f"Label   : {train['label'].iloc[i]} -> {train['label_3class'].iloc[i]}")

=== Exemples de nettoyage ===

--- Exemple 1 ---
Brut    : Says the Annies List political group supports third-trimester abortions on demand.
Nettoyé : annies list political group supports third trimester abortions demand
Label   : false -> fake

--- Exemple 2 ---
Brut    : When did the decline of coal start? It started when natural gas took off that started to begin in (President George W.) Bushs administration.
Nettoyé : when decline coal start started when natural gas took off started begin president george bushs administration
Label   : half-true -> nuanced

--- Exemple 3 ---
Brut    : Hillary Clinton agrees with John McCain "by voting to give George Bush the benefit of the doubt on Iran."
Nettoyé : hillary clinton agrees john mccain voting give george bush benefit doubt iran
Label   : mostly-true -> real


## 3. Distribution des labels après regroupement

In [4]:
ORDRE_3CLASS = ["fake", "nuanced", "real"]
COULEURS_3CLASS = ["#d73027", "#fdae61", "#1a9850"]

splits_data = {
    "Train": train,
    "Valid": valid,
    "Test" : test
}

rows = []
for nom, df in splits_data.items():
    counts = df["label_3class"].value_counts()
    for label in ORDRE_3CLASS:
        n = counts.get(label, 0)
        rows.append({
            "split": nom,
            "label": label,
            "count": n,
            "pct": round(n / len(df) * 100, 1)
        })

df_dist = pd.DataFrame(rows)

fig = px.bar(
    df_dist,
    x="split", y="pct", color="label",
    barmode="group",
    color_discrete_sequence=COULEURS_3CLASS,
    title="Distribution des 3 classes après regroupement (Train / Valid / Test)",
    labels={"pct": "Pourcentage (%)", "split": "Split", "label": "Classe"},
    text="pct"
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.show()

print("\n=== Table comparative ===")
print(df_dist.pivot(index="label", columns="split", values="pct").to_string())


=== Table comparative ===
split    Test  Train  Valid
label                      
fake     26.9   27.7   29.5
nuanced  37.6   36.8   37.8
real     35.4   35.5   32.7


**Lecture** : si les 3 classes sont proches en proportion sur les 3 splits,
notre évaluation sera fiable. Un écart > 5 points sur une classe
justifierait d'utiliser le F1-macro plutôt que l'accuracy.

## 4. Vérification des features de métadonnées

In [5]:
META_COLS = ["credibility_score", "lie_rate", "history_total", "is_politician"]

print("=== Statistiques des features de métadonnées (train) ===")
print(train[META_COLS].describe().round(3).to_string())

# Visualisation : credibility_score par classe
fig = px.box(
    train,
    x="label_3class", y="credibility_score",
    color="label_3class",
    color_discrete_sequence=COULEURS_3CLASS,
    category_orders={"label_3class": ORDRE_3CLASS},
    title="Distribution du credibility_score par classe (train)",
    labels={"credibility_score": "Score de crédibilité", "label_3class": "Classe"}
)
fig.update_layout(showlegend=False)
fig.show()

=== Statistiques des features de métadonnées (train) ===
       credibility_score   lie_rate  history_total  is_politician
count          10240.000  10240.000      10240.000      10240.000
mean               0.471      0.304         64.576          0.765
std                0.301      0.275        115.730          0.424
min                0.000      0.000          0.000          0.000
25%                0.270      0.083          2.000          1.000
50%                0.500      0.265         11.000          1.000
75%                0.667      0.454         65.000          1.000
max                1.000      1.000        473.000          1.000


**Ce qu'on vérifie ici** : si les boîtes sont bien séparées entre les 3 classes,
le `credibility_score` est une feature discriminante utile pour les modèles classiques.

**Biais à noter** : cette feature juge le *speaker*, pas la *déclaration*.
Un speaker habituellement fiable pourrait voir ses rares mensonges mal classifiés.
Ce point sera analysé dans la section éthique du rapport.

## 5. Construction des features TF-IDF + métadonnées

In [6]:
# --- TF-IDF seul (baseline) ---
tfidf = TfidfFeatures(max_features=20_000, ngram_range=(1, 2))

X_train_tfidf, y_train = get_X_y(train, tfidf, fit=True)
X_valid_tfidf, y_valid = get_X_y(valid, tfidf, fit=False)
X_test_tfidf,  y_test  = get_X_y(test,  tfidf, fit=False)

print(f"\nTF-IDF shapes :")
print(f"  Train : {X_train_tfidf.shape}")
print(f"  Valid : {X_valid_tfidf.shape}")
print(f"  Test  : {X_test_tfidf.shape}")

TF-IDF fit — vocabulaire : 9049 features, 10240 documents
X shape: (10240, 9049), y shape: (10240,), classes: (array([0, 1, 2]), array([2834, 3768, 3638]))
X shape: (1284, 9049), y shape: (1284,), classes: (array([0, 1, 2]), array([379, 485, 420]))
X shape: (1267, 9049), y shape: (1267,), classes: (array([0, 1, 2]), array([341, 477, 449]))

TF-IDF shapes :
  Train : (10240, 9049)
  Valid : (1284, 9049)
  Test  : (1267, 9049)


In [7]:
# --- TF-IDF + métadonnées (modèle principal) ---
tfidf_for_combined = TfidfFeatures(max_features=20_000, ngram_range=(1, 2))
combined = CombinedFeatures(text_features=tfidf_for_combined, scale_meta=True)

X_train_combined, _ = get_X_y(train, combined, fit=True)
X_valid_combined, _ = get_X_y(valid, combined, fit=False)
X_test_combined,  _ = get_X_y(test,  combined, fit=False)

print(f"\nCombined (TF-IDF + meta) shapes :")
print(f"  Train : {X_train_combined.shape}")
print(f"  Valid : {X_valid_combined.shape}")
print(f"  Test  : {X_test_combined.shape}")

TF-IDF fit — vocabulaire : 9049 features, 10240 documents
X shape: (10240, 9053), y shape: (10240,), classes: (array([0, 1, 2]), array([2834, 3768, 3638]))
X shape: (1284, 9053), y shape: (1284,), classes: (array([0, 1, 2]), array([379, 485, 420]))
X shape: (1267, 9053), y shape: (1267,), classes: (array([0, 1, 2]), array([341, 477, 449]))

Combined (TF-IDF + meta) shapes :
  Train : (10240, 9053)
  Valid : (1284, 9053)
  Test  : (1267, 9053)


## 6. Sauvegarde des données préprocessées

In [8]:
# Sauvegarde des DataFrames préprocessés
# (on garde uniquement les colonnes utiles pour alléger les fichiers)
COLS_TO_SAVE = [
    "id", "statement", "statement_clean", "label",
    "label_3class", "label_encoded",
    "speaker", "party", "subject",
    "credibility_score", "lie_rate", "history_total", "is_politician"
]

train[COLS_TO_SAVE].to_csv("data/processed/train_clean.csv", index=False)
valid[COLS_TO_SAVE].to_csv("data/processed/valid_clean.csv", index=False)
test[COLS_TO_SAVE].to_csv( "data/processed/test_clean.csv",  index=False)

# Sauvegarde du vectoriseur TF-IDF (pour le réutiliser dans 03_generalization.ipynb)
tfidf_for_combined.save("models/tfidf_vectorizer.pkl")

print("Sauvegarde terminée !")
print("  data/processed/train_clean.csv")
print("  data/processed/valid_clean.csv")
print("  data/processed/test_clean.csv")
print("  models/tfidf_vectorizer.pkl")

TF-IDF sauvegardé -> models/tfidf_vectorizer.pkl
Sauvegarde terminée !
  data/processed/train_clean.csv
  data/processed/valid_clean.csv
  data/processed/test_clean.csv
  models/tfidf_vectorizer.pkl


## 7. Vérification finale — top features TF-IDF par classe

On confirme ici que le TF-IDF a bien appris des features discriminantes,
cohérentes avec l'analyse exploratoire du notebook 01.

In [10]:
from sklearn.linear_model import LogisticRegression

# Entraîner une LR rapide pour identifier les features les plus importantes
# (pas d'optimisation ici, juste pour la visualisation)
lr_quick = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    C=1.0,
    solver="lbfgs",
    # multi_class="multinomial"
)
lr_quick.fit(X_train_tfidf, y_train)

feature_names = tfidf.get_feature_names()
label_names   = ["fake", "nuanced", "real"]
N_TOP = 10

print("=== Top features TF-IDF par classe ===")
for i, label in enumerate(label_names):
    coefs = lr_quick.coef_[i]
    top_idx = np.argsort(coefs)[::-1][:N_TOP]
    print(f"\n--- {label.upper()} ---")
    for idx in top_idx:
        print(f"   {feature_names[idx]:<30} coef: {coefs[idx]:.3f}")

=== Top features TF-IDF par classe ===

--- FAKE ---
   muslim                         coef: 1.524
   debunked                       coef: 1.409
   wisconsin                      coef: 1.403
   face                           coef: 1.303
   cabinet                        coef: 1.276
   vaccines                       coef: 1.209
   government                     coef: 1.187
   obama                          coef: 1.117
   jan                            coef: 1.094
   navy                           coef: 1.093

--- NUANCED ---
   bailout                        coef: 1.237
   put                            coef: 1.221
   standards                      coef: 1.210
   cut                            coef: 1.137
   manufacturing                  coef: 1.094
   because                        coef: 1.081
   hour                           coef: 1.069
   obama voted                    coef: 1.050
   billion year                   coef: 1.022
   increased                      coef: 1.009

--- REAL 

**Ce qu'on vérifie** : les features les plus discriminantes pour `fake`
doivent être des mots associés aux exagérations politiques ou aux
affirmations non vérifiables. Pour `real`, on attend des mots plus factuels.

Si les résultats semblent incohérents (noms propres dominant tout),
c'est un signe de sur-apprentissage sur les speakers — à documenter dans
l'analyse des biais.

---
**Prochaine étape** : notebook `03_models.ipynb` — entraînement complet
des modèles classiques (Logistic Regression, Random Forest, XGBoost)
et fine-tuning BERT.